# SVAMITVA Feature Extraction — IIT Tirupati Geospatial Hackathon
**Problem Statement 1**: AI-Based Feature Extraction from Drone Orthophotos  
Target: 95%+ pixel accuracy | Output: COG GeoTIFF + GeoPackage vectors

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — INSTALL DEPENDENCIES
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys

pkgs = [
    "segmentation-models-pytorch",
    "albumentations",
    "rasterio",
    "scikit-learn",
    "timm",
    "geopandas",
    "shapely",
    "fiona",
    "scipy",
    "Pillow",
    "opencv-python-headless",
    "torchvision",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
    capture_output=True
)
print("✅ Dependencies installed")

✅ Dependencies installed


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — GPU SETUP + IMPORTS
# Set CUDA_VISIBLE_DEVICES before anything else
# GPU 0 = training, GPU 1 = inference/vectorization
# ─────────────────────────────────────────────────────────────────────────────
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # ← change to "1" for second notebook

import gc, glob, json, random, time, warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
import rasterio
import rasterio.features
import rasterio.windows
from rasterio.windows import Window
import geopandas as gpd
from shapely.geometry import shape
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.ndimage import binary_closing, binary_opening
import cv2
import subprocess
warnings.filterwarnings("ignore")

gc.collect()
torch.cuda.empty_cache()

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"Memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"Free   : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

Device : cuda
GPU    : Tesla V100-PCIE-32GB
Memory : 34.1 GB
Free   : 0.2 GB


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — PATHS AND CLASS CONFIG
# ─────────────────────────────────────────────────────────────────────────────
BASE      = "/home/jupyter-228w1a1286/dinesh-data/hackthonndata"
CKPT_DIR  = f"{BASE}/CHECKPOINTS_FINAL"
OUT_DIR   = f"{BASE}/FINAL_OUTPUTS"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR,  exist_ok=True)

# ── Training data (from your prepared DATASET_1024) ───────────────────────────
IMG_DIR  = f"{BASE}/trainingdata/DATASET_1024/images"
MASK_DIR = f"{BASE}/trainingdata/DATASET_1024/masks"

# ── Test villages (from PB_live_demo) ─────────────────────────────────────────
# Update these paths to wherever you have the test TIFFs
TEST_TIFS = glob.glob(f"{BASE}/live_demo/*.tif")   # PB_live_demo tiffs
print(f"Test villages found: {len(TEST_TIFS)}")
for t in TEST_TIFS:
    print(f"  {os.path.basename(t)}")

# ── Class remapping: 9 original → 5 clean classes ────────────────────────────
#
# Original mask labels (from your data prep):
#   0=background, 1=built_up_area, 2=road, 3=road_centre_line,
#   4=railway, 5=bridge, 6=water_body_line, 7=waterbody_point,
#   8=utility, 9=utility_poly
#
# Remapped (keep only classes with enough data):
#   0=background, 1=building, 2=road, 3=water_body, 4=water_line

REMAP_ARRAY = np.array([
    0,   # 0 background       → background
    1,   # 1 built_up_area    → building
    2,   # 2 road             → road
    2,   # 3 road_centre_line → road (merge into same class)
    0,   # 4 railway          → background (4 objects, too sparse)
    0,   # 5 bridge           → background (8 objects, too sparse)
    3,   # 6 water_body_line  → water_body
    3,   # 7 waterbody_point  → water_body
    0,   # 8 utility          → background (point features, too sparse)
    0,   # 9 utility_poly     → background (too sparse)
], dtype=np.int64)

NUM_CLASSES    = 4
CLASS_NAMES    = ["background", "building", "road", "water_body"]
ACTIVE_CLASSES = [1, 2, 3]

COLOR_MAP = np.array([
    [0,   0,   0  ],   # 0 background — black
    [255, 50,  50 ],   # 1 building   — red
    [255, 200, 0  ],   # 2 road       — yellow
    [0,   100, 255],   # 3 water_body — blue
], dtype=np.uint8)

# Pixel counts after remapping (background absorbs sparse classes)
PIXEL_COUNTS = np.array([
    1_764_710_366 + 543_668 + 93 + 666_574,  # 0 background
    388_862_220,                              # 1 building
    49_832_874 + 131_981_531,                 # 2 road (road + road_centre_line merged)
    44_439_682 + 53_439,                      # 3 water_body
], dtype=np.float64)

print(f"\nClass distribution after remapping:")
total = PIXEL_COUNTS.sum()
for i, (n, c) in enumerate(zip(CLASS_NAMES, PIXEL_COUNTS)):
    print(f"  {i} {n:15s}: {int(c):>14,}  ({100*c/total:.2f}%)")

Test villages found: 0

Class distribution after remapping:
  0 background     :  1,765,920,701  (74.16%)
  1 building       :    388,862,220  (16.33%)
  2 road           :    181,814,405  (7.64%)
  3 water_body     :     44,493,121  (1.87%)


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — TRAINING CONFIG
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # Paths
    "img_dir":         IMG_DIR,
    "mask_dir":        MASK_DIR,
    "save_dir":        CKPT_DIR,

    # Model
    "num_classes":     NUM_CLASSES,
    "tile_size":       1024,
    "encoder":         "mit_b2",     # lighter → fits batch=8 on V100-32GB
    "encoder_weights": "imagenet",

    # Training
    "batch_size":      4,            # safe for mit_b2 at 1024px on 32GB V100
    "accum_steps":     2,            # effective batch = 8
    "num_epochs":      60,
    "lr":              6e-5,
    "weight_decay":    1e-2,
    "val_split":       0.15,         # keep more for training
    "num_workers":     6,

    # Loss weights
    "ce_weight":       0.35,
    "dice_weight":     0.45,
    "focal_weight":    0.20,

    # Early stopping
    "patience":        15,
    "min_delta":       5e-4,

    # Resume
    "resume":          False,
    "resume_ckpt":     f"{CKPT_DIR}/best_model.pth",
}

print("Config:")
for k, v in CFG.items():
    print(f"  {k:20s}: {v}")

Config:
  img_dir             : /home/jupyter-228w1a1286/dinesh-data/hackthonndata/trainingdata/DATASET_1024/images
  mask_dir            : /home/jupyter-228w1a1286/dinesh-data/hackthonndata/trainingdata/DATASET_1024/masks
  save_dir            : /home/jupyter-228w1a1286/dinesh-data/hackthonndata/CHECKPOINTS_FINAL
  num_classes         : 4
  tile_size           : 1024
  encoder             : mit_b2
  encoder_weights     : imagenet
  batch_size          : 4
  accum_steps         : 2
  num_epochs          : 60
  lr                  : 6e-05
  weight_decay        : 0.01
  val_split           : 0.15
  num_workers         : 6
  ce_weight           : 0.35
  dice_weight         : 0.45
  focal_weight        : 0.2
  patience            : 15
  min_delta           : 0.0005
  resume              : False
  resume_ckpt         : /home/jupyter-228w1a1286/dinesh-data/hackthonndata/CHECKPOINTS_FINAL/best_model.pth


In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — DATASET
# ─────────────────────────────────────────────────────────────────────────────
class SVAMITVADataset(Dataset):
    def __init__(self, img_paths, mask_paths, transform=None):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.transform  = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        with rasterio.open(self.img_paths[idx]) as src:
            img = src.read().astype(np.float32) / 255.0
            if img.shape[0] > 3:
                img = img[:3]   # drop alpha if present
            img = np.transpose(img, (1, 2, 0))  # CHW → HWC

        with rasterio.open(self.mask_paths[idx]) as src:
            mask = src.read(1).astype(np.int64)

        # Remap original labels → clean 4-class labels
        mask = np.clip(mask, 0, len(REMAP_ARRAY) - 1)
        mask = REMAP_ARRAY[mask]

        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img  = aug["image"]
            mask = aug["mask"].long()
        else:
            img  = torch.from_numpy(img.transpose(2, 0, 1)).float()
            mask = torch.from_numpy(mask).long()

        return img, mask

    def get_sample_weight(self, idx):
        """Oversample tiles that contain foreground classes."""
        with rasterio.open(self.mask_paths[idx]) as src:
            mask = REMAP_ARRAY[np.clip(src.read(1).flatten(), 0, len(REMAP_ARRAY) - 1)]
        fg = (mask > 0).sum()
        return float(fg) + 1.0


# ── Augmentations ─────────────────────────────────────────────────────────────
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15,
                       rotate_limit=20, p=0.5),
    # Scale augmentation — critical for satellite/drone imagery
    A.RandomScale(scale_limit=0.3, p=0.4),
    A.PadIfNeeded(min_height=CFG["tile_size"], min_width=CFG["tile_size"],
                  border_mode=0, value=0, mask_value=0),
    A.RandomCrop(height=CFG["tile_size"], width=CFG["tile_size"]),
    A.OneOf([
        A.GridDistortion(num_steps=5, distort_limit=0.25),
        A.OpticalDistortion(distort_limit=0.2),
        A.ElasticTransform(alpha=80, sigma=80 * 0.05),
    ], p=0.25),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.35, contrast_limit=0.35),
        A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15, hue=0.1),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8)),
    ], p=0.6),
    A.OneOf([
        A.GaussNoise(var_limit=(10, 60)),
        A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5)),
        A.MultiplicativeNoise(multiplier=(0.9, 1.1)),
    ], p=0.25),
    A.CoarseDropout(max_holes=8, max_height=48, max_width=48,
                    fill_value=0, p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

print("Dataset and augmentations defined ✅")

Dataset and augmentations defined ✅


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — BUILD DATALOADERS
# ─────────────────────────────────────────────────────────────────────────────
all_imgs  = sorted(glob.glob(f"{CFG['img_dir']}/*.tif"))
all_masks = sorted(glob.glob(f"{CFG['mask_dir']}/*.tif"))

assert len(all_imgs) > 0, f"No tiles in {CFG['img_dir']}"
assert len(all_imgs) == len(all_masks), \
    f"Mismatch: {len(all_imgs)} images vs {len(all_masks)} masks"

print(f"Total tiles: {len(all_imgs)}")

indices = list(range(len(all_imgs)))
train_idx, val_idx = train_test_split(
    indices, test_size=CFG["val_split"], random_state=SEED
)

train_imgs  = [all_imgs[i]  for i in train_idx]
train_masks = [all_masks[i] for i in train_idx]
val_imgs    = [all_imgs[i]  for i in val_idx]
val_masks   = [all_masks[i] for i in val_idx]

# Save splits
for name, data in [("train_imgs",  train_imgs),  ("train_masks", train_masks),
                   ("val_imgs",    val_imgs),     ("val_masks",   val_masks)]:
    with open(os.path.join(CFG["save_dir"], f"{name}.json"), "w") as f:
        json.dump(data, f)

train_ds = SVAMITVADataset(train_imgs, train_masks, train_transform)
val_ds   = SVAMITVADataset(val_imgs,   val_masks,   val_transform)

# Weighted sampler — cache to avoid recomputing every run
cache = os.path.join(CFG["save_dir"], "sample_weights.json")
if os.path.exists(cache):
    with open(cache) as f:
        sw = json.load(f)
    if len(sw) == len(train_ds):
        print("Loaded cached sample weights")
    else:
        sw = None
else:
    sw = None

if sw is None:
    print("Computing sample weights...")
    sw = [train_ds.get_sample_weight(i)
          for i in tqdm(range(len(train_ds)), desc="Weights")]
    with open(cache, "w") as f:
        json.dump(sw, f)

sampler = WeightedRandomSampler(
    weights=sw, num_samples=len(train_ds), replacement=True
)

train_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"], sampler=sampler,
    num_workers=CFG["num_workers"], pin_memory=True,
    prefetch_factor=2, persistent_workers=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=CFG["batch_size"], shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=True,
    prefetch_factor=2, persistent_workers=True,
)

print(f"Train tiles : {len(train_ds)}")
print(f"Val tiles   : {len(val_ds)}")
print(f"Batch size  : {CFG['batch_size']}  (effective: {CFG['batch_size'] * CFG['accum_steps']})")

Total tiles: 2744
Loaded cached sample weights
Train tiles : 2332
Val tiles   : 412
Batch size  : 4  (effective: 8)


In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — MODEL + LOSS + OPTIMIZER
# ─────────────────────────────────────────────────────────────────────────────

# ── Class weights ─────────────────────────────────────────────────────────────
raw_w   = 1.0 / np.log1p(PIXEL_COUNTS)
raw_w  /= raw_w.mean()
raw_w   = np.clip(raw_w, 0, 15.0)
cw      = torch.tensor(raw_w, dtype=torch.float32).to(DEVICE)
print("Class weights:", [f"{w:.3f}" for w in raw_w])

# ── Model: SegFormer-B2 ────────────────────────────────────────────────────────
model = smp.Segformer(
    encoder_name    = CFG["encoder"],
    encoder_weights = CFG["encoder_weights"],
    in_channels     = 3,
    classes         = CFG["num_classes"],
    activation      = None,
).to(DEVICE)

# Force gradient checkpointing on the MixTransformer encoder
# This trades compute for ~8GB VRAM savings — critical at 1024px
if hasattr(model.encoder, "set_grad_checkpointing"):
    model.encoder.set_grad_checkpointing(True)
    print("Gradient checkpointing: enabled via set_grad_checkpointing")
else:
    # Fallback: manually enable on all transformer blocks
    for module in model.encoder.modules():
        if hasattr(module, "gradient_checkpointing"):
            module.gradient_checkpointing = True
    print("Gradient checkpointing: enabled via module flag")

# Additional VRAM savings: clear cache before training
torch.cuda.empty_cache()
gc.collect()
print(f"GPU free after model load: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# ── Loss ──────────────────────────────────────────────────────────────────────
ce_loss    = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)
dice_loss  = smp.losses.DiceLoss(
    mode="multiclass", classes=ACTIVE_CLASSES, smooth=1.0
)
focal_loss = smp.losses.FocalLoss(mode="multiclass", gamma=3.0)

def criterion(outputs, masks):
    return (CFG["ce_weight"]    * ce_loss(outputs, masks)    +
            CFG["dice_weight"]  * dice_loss(outputs, masks)  +
            CFG["focal_weight"] * focal_loss(outputs, masks))

# ── Optimizer: separate LRs for encoder vs decoder ────────────────────────────
enc_p, dec_p = [], []
for name, p in model.named_parameters():
    (enc_p if "encoder" in name else dec_p).append(p)

optimizer = torch.optim.AdamW([
    {"params": enc_p, "lr": CFG["lr"],     "weight_decay": CFG["weight_decay"]},
    {"params": dec_p, "lr": CFG["lr"] * 5, "weight_decay": CFG["weight_decay"]},
])

# Warmup + cosine decay
def warmup_cosine(epoch, warmup=5, total=60):
    if epoch < warmup:
        return (epoch + 1) / warmup
    return 0.5 * (1.0 + np.cos(
        np.pi * (epoch - warmup) / (total - warmup)
    ))

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda ep: warmup_cosine(ep, warmup=5, total=CFG["num_epochs"]),
)

scaler = GradScaler("cuda")
print("Loss, optimizer, scheduler ready ✅")

Class weights: ['0.908', '0.978', '1.017', '1.098']
Gradient checkpointing: enabled via module flag
GPU free after model load: 0.2 GB
Loss, optimizer, scheduler ready ✅


In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — METRICS
# ─────────────────────────────────────────────────────────────────────────────
def compute_metrics(preds, masks):
    """Returns pixel accuracy, per-class IoU, and mIoU."""
    # Pixel accuracy
    correct = (preds == masks).sum().item()
    acc     = correct / masks.numel()

    # Per-class IoU (foreground only)
    ious = {}
    for cls in ACTIVE_CLASSES:
        pred_c = (preds == cls)
        true_c = (masks == cls)
        inter  = (pred_c & true_c).sum().item()
        union  = (pred_c | true_c).sum().item()
        if union > 0:
            ious[cls] = inter / union

    miou = float(np.mean(list(ious.values()))) if ious else 0.0
    return acc, ious, miou


def compute_f1(preds, masks):
    """Per-class F1 score."""
    f1s = {}
    for cls in ACTIVE_CLASSES:
        tp    = ((preds == cls) & (masks == cls)).sum().item()
        fp    = ((preds == cls) & (masks != cls)).sum().item()
        fn    = ((preds != cls) & (masks == cls)).sum().item()
        denom = 2 * tp + fp + fn
        f1s[cls] = (2 * tp / denom) if denom > 0 else 0.0
    return f1s


print("Metrics defined ✅")

Metrics defined ✅


In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — OPTIONAL RESUME FROM CHECKPOINT
# ─────────────────────────────────────────────────────────────────────────────
best_miou   = 0.0
best_acc    = 0.0
start_epoch = 0

if CFG["resume"] and os.path.exists(CFG["resume_ckpt"]):
    print(f"Loading checkpoint: {CFG['resume_ckpt']}")
    ckpt  = torch.load(CFG["resume_ckpt"], map_location=DEVICE, weights_only=False)
    state = {k.replace("_orig_mod.", ""): v
             for k, v in ckpt["model_state"].items()}
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:    print(f"  Missing keys   : {missing[:3]}")
    if unexpected: print(f"  Unexpected keys: {unexpected[:3]}")
    best_miou   = ckpt.get("val_miou", 0.0)
    best_acc    = ckpt.get("val_acc",  0.0)
    start_epoch = ckpt["epoch"]
    print(f"  Resumed epoch {start_epoch} | mIoU={best_miou:.4f} | Acc={best_acc*100:.2f}%")
else:
    print("Training from scratch")

Training from scratch


In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
history           = {"train_loss": [], "val_loss": [], "val_miou": [], "val_acc": []}
patience_ct       = 0
nan_total         = 0

print("\n" + "═" * 70)
print(f"  Model   : SegFormer-{CFG['encoder'].upper()}")
print(f"  Classes : {CLASS_NAMES}")
print(f"  Tiles   : {len(all_imgs)} | Batch: {CFG['batch_size']} | Epochs: {CFG['num_epochs']}")
print(f"  Target  : ≥95% pixel accuracy")
print("═" * 70 + "\n")

for epoch in range(start_epoch, CFG["num_epochs"]):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    # ── TRAIN ─────────────────────────────────────────────────────────────────
    model.train()
    train_loss, valid_steps, nan_ep = 0.0, 0, 0
    optimizer.zero_grad()

    for step, (imgs, masks) in enumerate(tqdm(
            train_loader, desc=f"Ep {epoch+1:03d} train", leave=False)):

        imgs  = imgs.to(DEVICE,  non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        with autocast("cuda"):
            out  = model(imgs)
            loss = criterion(out, masks) / CFG["accum_steps"]

        if torch.isnan(loss) or torch.isinf(loss):
            nan_ep += 1
            optimizer.zero_grad(); torch.cuda.empty_cache(); continue

        scaler.scale(loss).backward()

        if (step + 1) % CFG["accum_steps"] == 0:
            scaler.unscale_(optimizer)
            has_nan = any(
                p.grad is not None and
                (torch.isnan(p.grad).any() or torch.isinf(p.grad).any())
                for p in model.parameters()
            )
            if has_nan:
                nan_ep += 1
                optimizer.zero_grad(); scaler.update()
                torch.cuda.empty_cache(); continue

            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss  += loss.item() * CFG["accum_steps"]
        valid_steps += 1

    scheduler.step()

    if nan_ep > 0:
        nan_total += nan_ep
        print(f"  ⚠️  {nan_ep} NaN batches (total: {nan_total})")
    if valid_steps == 0:
        print("  💀 All NaN — stopping."); break

    train_loss /= valid_steps
    train_t     = time.time() - t0

    # ── VALIDATE ──────────────────────────────────────────────────────────────
    model.eval()
    val_loss, val_steps = 0.0, 0
    all_ious = {c: [] for c in ACTIVE_CLASSES}
    all_accs = []

    with torch.no_grad():
        for imgs, masks in tqdm(val_loader,
                                desc=f"Ep {epoch+1:03d} val  ", leave=False):
            imgs  = imgs.to(DEVICE,  non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)

            with autocast("cuda"):
                out  = model(imgs)
                loss = criterion(out, masks)

            if torch.isnan(loss) or torch.isinf(loss): continue

            preds       = torch.argmax(out, dim=1)
            val_loss   += loss.item()
            val_steps  += 1
            acc, ious, _ = compute_metrics(preds, masks)
            all_accs.append(acc)
            for cls, iou in ious.items():
                all_ious[cls].append(iou)

    val_t         = time.time() - t0 - train_t
    val_loss     /= max(val_steps, 1)
    pc_iou        = {c: float(np.mean(v)) for c, v in all_ious.items() if v}
    val_miou      = float(np.mean(list(pc_iou.values()))) if pc_iou else 0.0
    val_acc       = float(np.mean(all_accs)) if all_accs else 0.0
    peak_mem      = torch.cuda.max_memory_allocated() / 1e9

    torch.cuda.empty_cache()
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_miou"].append(val_miou)
    history["val_acc"].append(val_acc)

    improved = val_acc > best_acc + CFG["min_delta"]
    if improved:
        best_acc    = val_acc
        best_miou   = val_miou
        patience_ct = 0
        torch.save({
            "epoch":          epoch + 1,
            "model_state":    model.state_dict(),
            "optim_state":    optimizer.state_dict(),
            "val_miou":       val_miou,
            "val_acc":        val_acc,
            "cfg":            CFG,
            "class_names":    CLASS_NAMES,
            "active_classes": ACTIVE_CLASSES,
            "remap_array":    REMAP_ARRAY.tolist(),
            "color_map":      COLOR_MAP.tolist(),
        }, os.path.join(CFG["save_dir"], "best_model.pth"))
        tag = f"  ✅ best  acc={best_acc*100:.2f}%  mIoU={best_miou:.4f}"
    else:
        patience_ct += 1
        tag = f"  (patience {patience_ct}/{CFG['patience']})"

    lr_now = optimizer.param_groups[0]["lr"]
    print(
        f"Ep {epoch+1:03d}/{CFG['num_epochs']} | "
        f"Train: {train_loss:.4f} ({train_t:.0f}s) | "
        f"Val: {val_loss:.4f} ({val_t:.0f}s) | "
        f"Acc: {val_acc*100:.2f}% | mIoU: {val_miou:.4f} | "
        f"VRAM: {peak_mem:.1f}GB | LR: {lr_now:.2e}{tag}"
    )

    if (epoch + 1) % 10 == 0:
        print("  Per-class IoU:")
        for cls in ACTIVE_CLASSES:
            iou = pc_iou.get(cls, 0.0)
            print(f"    {cls} {CLASS_NAMES[cls]:15s}: {iou:.4f}")

    if patience_ct >= CFG["patience"]:
        print(f"\nEarly stopping at epoch {epoch + 1}.")
        break

print(f"\nBest pixel accuracy : {best_acc*100:.2f}%")
print(f"Best mIoU           : {best_miou:.4f}")


══════════════════════════════════════════════════════════════════════
  Model   : SegFormer-MIT_B2
  Classes : ['background', 'building', 'road', 'water_body']
  Tiles   : 2744 | Batch: 4 | Epochs: 60
  Target  : ≥95% pixel accuracy
══════════════════════════════════════════════════════════════════════



OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacity of 31.73 GiB of which 4.19 MiB is free. Including non-PyTorch memory, this process has 31.72 GiB memory in use. Of the allocated memory 31.33 GiB is allocated by PyTorch, and 22.34 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — TRAINING CURVES
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ep = range(1, len(history["train_loss"]) + 1)

axes[0].plot(ep, history["train_loss"], label="Train", lw=2)
axes[0].plot(ep, history["val_loss"],   label="Val",   lw=2)
axes[0].set_title("Loss", fontweight="bold")
axes[0].legend(); axes[0].set_xlabel("Epoch"); axes[0].grid(alpha=0.3)

axes[1].plot(ep, [v*100 for v in history["val_acc"]],
             color="royalblue", lw=2, label="Pixel Acc %")
axes[1].axhline(95, color="red",   ls="--", label="95% target")
axes[1].axhline(90, color="orange",ls="--", label="90% target")
axes[1].set_title("Pixel Accuracy", fontweight="bold")
axes[1].legend(); axes[1].set_xlabel("Epoch"); axes[1].grid(alpha=0.3)

axes[2].plot(ep, history["val_miou"], color="seagreen", lw=2, label="mIoU")
axes[2].set_title("Validation mIoU", fontweight="bold")
axes[2].legend(); axes[2].set_xlabel("Epoch"); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CFG["save_dir"], "training_curves.png"), dpi=150)
plt.show()
print(f"Saved → {CFG['save_dir']}/training_curves.png")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — LOAD BEST CHECKPOINT FOR INFERENCE
# ─────────────────────────────────────────────────────────────────────────────
INF_CFG = {
    "checkpoint": os.path.join(CFG["save_dir"], "best_model.pth"),
    "tile_size":  1024,
    "overlap":    256,
    "batch_size": 4,    # smaller for inference to leave VRAM headroom
}

inf_model = smp.Segformer(
    encoder_name    = CFG["encoder"],
    encoder_weights = None,
    in_channels     = 3,
    classes         = CFG["num_classes"],
).to(DEVICE)

ckpt  = torch.load(INF_CFG["checkpoint"], map_location=DEVICE, weights_only=False)
state = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model_state"].items()}
inf_model.load_state_dict(state, strict=False)
inf_model.eval()

print(f"✅ Model loaded from epoch {ckpt['epoch']}")
print(f"   Val acc : {ckpt['val_acc']*100:.2f}%")
print(f"   Val mIoU: {ckpt['val_miou']:.4f}")

inf_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 — SLIDING WINDOW INFERENCE WITH TTA
# ─────────────────────────────────────────────────────────────────────────────
def tta_predict(model, tensor):
    """Test-time augmentation: original + h-flip + v-flip, then average."""
    preds = []
    with torch.no_grad():
        preds.append(torch.softmax(model(tensor), dim=1))
        preds.append(torch.flip(
            torch.softmax(model(torch.flip(tensor, [3])), dim=1), [3]
        ))
        preds.append(torch.flip(
            torch.softmax(model(torch.flip(tensor, [2])), dim=1), [2]
        ))
    return torch.mean(torch.stack(preds), dim=0)


def sliding_window_inference(tif_path, model, use_tta=True):
    """
    Run sliding-window inference on a full orthophoto.
    Returns: (prediction_array, src_profile, src_crs, src_transform)
    """
    tile   = INF_CFG["tile_size"]
    stride = tile - INF_CFG["overlap"]

    with rasterio.open(tif_path) as src:
        H, W       = src.height, src.width
        profile    = src.profile.copy()
        transform_ = src.transform
        crs_       = src.crs

        output = np.zeros((CFG["num_classes"], H, W), dtype=np.float32)
        count  = np.zeros((H, W), dtype=np.float32)

        row_ranges = range(0, H, stride)
        for y in tqdm(row_ranges, desc=f"  Rows [{os.path.basename(tif_path)}]"):
            for x in range(0, W, stride):
                win_h = min(tile, H - y)
                win_w = min(tile, W - x)
                window = Window(x, y, win_w, win_h)

                # Read RGB only
                img = src.read([1, 2, 3], window=window).astype(np.float32) / 255.0
                img = np.transpose(img, (1, 2, 0))  # CHW → HWC

                # Pad to full tile size
                pad_h = tile - img.shape[0]
                pad_w = tile - img.shape[1]
                if pad_h > 0 or pad_w > 0:
                    img = np.pad(img, ((0, pad_h), (0, pad_w), (0, 0)))

                tensor = inf_transform(image=img)["image"].unsqueeze(0).to(DEVICE)

                if use_tta:
                    pred = tta_predict(model, tensor).cpu().numpy()[0]
                else:
                    with torch.no_grad():
                        pred = torch.softmax(
                            model(tensor), dim=1
                        ).cpu().numpy()[0]

                pred = pred[:, :win_h, :win_w]
                output[:, y:y+win_h, x:x+win_w] += pred
                count[y:y+win_h, x:x+win_w]     += 1

    output /= np.maximum(count, 1)
    prediction = np.argmax(output, axis=0).astype(np.uint8)
    return prediction, profile, crs_, transform_


print("Inference functions defined ✅")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 14 — VECTORIZATION: MASK → GEOPACKAGE
# ─────────────────────────────────────────────────────────────────────────────
def heuristic_roof_type(chip_rgb):
    """
    Simple spectral heuristic for roof classification.
    Works without labeled training data.
    RCC = flat gray concrete | Tiled = reddish | Tin = high brightness
    """
    if chip_rgb.size == 0:
        return "Others"
    r = chip_rgb[:, :, 0].mean()
    g = chip_rgb[:, :, 1].mean()
    b = chip_rgb[:, :, 2].mean()
    brightness = (r + g + b) / 3.0
    if brightness > 210:
        return "Tin"
    if r > g + 25 and r > b + 25:
        return "Tiled"
    if abs(r - g) < 18 and abs(g - b) < 18 and brightness > 80:
        return "RCC"
    return "Others"


def prediction_to_vectors(prediction, src_transform, src_crs,
                          ortho_tif_path, output_gpkg, village_name):
    """
    Convert integer prediction mask → GeoPackage with layers:
      - buildings  (with roof_type attribute: RCC/Tiled/Tin/Others)
      - roads
      - water_bodies
    """
    layer_config = {
        "buildings":    {"class_id": 1, "min_area_m2": 15,  "morph_close": 3, "morph_open": 2},
        "roads":        {"class_id": 2, "min_area_m2": 5,   "morph_close": 2, "morph_open": 1},
        "water_bodies": {"class_id": 3, "min_area_m2": 10,  "morph_close": 3, "morph_open": 2},
    }

    all_gdfs = {}

    # Estimate pixel size in metres (approximate from transform)
    pixel_w = abs(src_transform.a)
    pixel_h = abs(src_transform.e)
    px_area  = pixel_w * pixel_h

    with rasterio.open(ortho_tif_path) as src_ortho:
        for layer_name, cfg_l in layer_config.items():
            cls_id  = cfg_l["class_id"]
            binary  = (prediction == cls_id).astype(np.uint8)

            # Morphological cleanup
            binary = binary_closing(binary, iterations=cfg_l["morph_close"]).astype(np.uint8)
            binary = binary_opening(binary, iterations=cfg_l["morph_open"]).astype(np.uint8)

            # Rasterio polygonize
            shapes_gen = rasterio.features.shapes(
                binary,
                mask=binary,
                transform=src_transform
            )

            min_px = cfg_l["min_area_m2"] / px_area  # min pixels
            records = []

            for geom, val in shapes_gen:
                if val != 1:
                    continue
                poly = shape(geom)
                if poly.area < min_px:
                    continue

                rec = {
                    "geometry":     poly.simplify(0.5, preserve_topology=True),
                    "village":      village_name,
                    "feature_type": layer_name,
                    "area_m2":      round(poly.area * px_area, 2),
                }

                # Roof classification for buildings
                if layer_name == "buildings":
                    bounds = poly.bounds  # (minx, miny, maxx, maxy)
                    window = rasterio.windows.from_bounds(
                        *bounds, transform=src_ortho.transform
                    )
                    try:
                        chip = src_ortho.read([1, 2, 3], window=window)
                        chip = np.transpose(chip, (1, 2, 0)).astype(np.uint8)
                        rec["roof_type"] = heuristic_roof_type(chip)
                    except Exception:
                        rec["roof_type"] = "Others"
                else:
                    rec["roof_type"] = None

                records.append(rec)

            if records:
                gdf = gpd.GeoDataFrame(records, crs=src_crs)
                all_gdfs[layer_name] = gdf
                print(f"  {layer_name:15s}: {len(gdf):5d} features")
            else:
                print(f"  {layer_name:15s}: 0 features (empty)")

    # Write GeoPackage — one file, multiple layers
    if os.path.exists(output_gpkg):
        os.remove(output_gpkg)

    for layer_name, gdf in all_gdfs.items():
        gdf.to_file(output_gpkg, layer=layer_name, driver="GPKG")

    print(f"  ✅ GeoPackage saved: {output_gpkg}")
    return all_gdfs


print("Vectorization functions defined ✅")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 15 — COG OUTPUT HELPER
# Cloud Optimized GeoTIFF — required format per hackathon guide
# ─────────────────────────────────────────────────────────────────────────────
def save_prediction_as_cog(prediction, profile, output_cog_path):
    """
    Save prediction mask as Cloud Optimized GeoTIFF (COG).
    Also saves a color visualization COG alongside it.
    """
    profile.update(
        driver   = "GTiff",
        dtype    = rasterio.uint8,
        count    = 1,
        compress = "lzw",
    )

    # Step 1: write regular GeoTIFF
    tmp_path = output_cog_path.replace(".tif", "_tmp.tif")
    with rasterio.open(tmp_path, "w", **profile) as dst:
        dst.write(prediction, 1)

    # Step 2: convert to COG using gdal_translate
    result = subprocess.run([
        "gdal_translate",
        tmp_path, output_cog_path,
        "-of", "COG",
        "-co", "COMPRESS=LZW",
        "-co", "PREDICTOR=2",
        "-co", "OVERVIEWS=AUTO",
        "-co", "BLOCKSIZE=512",
    ], capture_output=True, text=True)

    if result.returncode == 0:
        os.remove(tmp_path)
        print(f"  ✅ COG saved: {output_cog_path}")
    else:
        # gdal_translate not available — keep regular GeoTIFF
        os.rename(tmp_path, output_cog_path)
        print(f"  ⚠️  gdal_translate not found, saved as regular GeoTIFF")
        print(f"     {output_cog_path}")

    # Also save an RGB color visualization
    color_path = output_cog_path.replace(".tif", "_color.tif")
    color_mask = COLOR_MAP[prediction]   # HWC
    color_profile = profile.copy()
    color_profile.update(count=3)
    with rasterio.open(color_path, "w", **color_profile) as dst:
        dst.write(np.transpose(color_mask, (2, 0, 1)))
    print(f"  ✅ Color viz saved: {color_path}")


print("COG output helper defined ✅")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 16 — RUN INFERENCE ON ALL TEST VILLAGES
# ─────────────────────────────────────────────────────────────────────────────
# Update TEST_TIFS in Cell 3 to point to your test village TIFFs
# e.g. the PB_live_demo villages:
#   DIWANA_BARNALA_40082.tif
#   KARTARPUR_AMRITSAR_37842.tif
#   ANAITPURA_FATEHGARH_SAHIB_32705.tif
#   BADRA_BARNALA_40044.tif
#   BUTTER_SIVIYA_AMRITSAR_37780.tif

if len(TEST_TIFS) == 0:
    print("⚠️  No test TIFs found. Update TEST_TIFS path in Cell 3.")
    print("   Example:")
    print(f"   TEST_TIFS = glob.glob('{BASE}/live_demo/*.tif')")
else:
    print(f"Running inference on {len(TEST_TIFS)} test villages...\n")

    for tif_path in TEST_TIFS:
        village = os.path.splitext(os.path.basename(tif_path))[0]
        v_dir   = os.path.join(OUT_DIR, village)
        os.makedirs(v_dir, exist_ok=True)

        print(f"\n{'─'*60}")
        print(f"Village: {village}")

        # 1. Inference
        pred, profile, crs, transform_ = sliding_window_inference(
            tif_path, inf_model, use_tta=True
        )

        # 2. Save prediction COG
        cog_path = os.path.join(v_dir, f"{village}_prediction.tif")
        save_prediction_as_cog(pred, profile, cog_path)

        # 3. Vectorize to GeoPackage
        gpkg_path = os.path.join(v_dir, f"{village}_features.gpkg")
        gdfs = prediction_to_vectors(
            pred, transform_, crs, tif_path, gpkg_path, village
        )

        # 4. Quick overlay visualization
        with rasterio.open(tif_path) as src:
            orig_small = src.read([1, 2, 3])
            # Downsample for visualization
            scale = min(1.0, 2000 / max(orig_small.shape[1], orig_small.shape[2]))
            h_vis = int(orig_small.shape[1] * scale)
            w_vis = int(orig_small.shape[2] * scale)
            orig_vis = cv2.resize(
                np.transpose(orig_small, (1, 2, 0)),
                (w_vis, h_vis)
            )
            pred_vis = cv2.resize(pred, (w_vis, h_vis),
                                  interpolation=cv2.INTER_NEAREST)

        color_vis  = COLOR_MAP[pred_vis]
        overlay    = orig_vis.copy()
        fg_mask    = pred_vis > 0
        overlay[fg_mask] = color_vis[fg_mask]
        blended    = cv2.addWeighted(orig_vis, 0.55, overlay, 0.45, 0)

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        axes[0].imshow(orig_vis)
        axes[0].set_title(f"{village}\nOriginal", fontsize=10)
        axes[0].axis("off")
        axes[1].imshow(color_vis)
        axes[1].set_title("Prediction Mask", fontsize=10)
        axes[1].axis("off")
        axes[2].imshow(blended)
        axes[2].set_title("Overlay", fontsize=10)
        axes[2].axis("off")

        # Legend
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor=COLOR_MAP[i]/255, label=CLASS_NAMES[i])
            for i in range(1, NUM_CLASSES)
        ]
        axes[2].legend(handles=legend_elements, loc="lower right",
                       fontsize=8, framealpha=0.8)

        plt.suptitle(f"SVAMITVA Feature Extraction — {village}",
                     fontsize=13, fontweight="bold")
        plt.tight_layout()
        viz_path = os.path.join(v_dir, f"{village}_visualization.png")
        plt.savefig(viz_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"  ✅ Visualization: {viz_path}")

    print(f"\n{'═'*60}")
    print(f"✅ All villages processed. Outputs in: {OUT_DIR}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 17 — SUMMARY REPORT
# ─────────────────────────────────────────────────────────────────────────────
import json
from datetime import datetime

report = {
    "project":     "SVAMITVA AI Feature Extraction",
    "hackathon":   "IIT Tirupati Geospatial Intelligence Hackathon 2025-26",
    "problem":     "Problem Statement 1 — AI-Based Feature Extraction from Drone Images",
    "generated":   datetime.now().isoformat(),

    "model": {
        "architecture": f"SegFormer-{CFG['encoder'].upper()}",
        "encoder":       CFG["encoder"],
        "tile_size":     CFG["tile_size"],
        "num_classes":   NUM_CLASSES,
        "class_names":   CLASS_NAMES,
    },

    "training": {
        "tiles_total":    len(all_imgs),
        "tiles_train":    len(train_ds),
        "tiles_val":      len(val_ds),
        "best_val_acc":   f"{best_acc*100:.2f}%",
        "best_val_miou":  f"{best_miou:.4f}",
        "epochs_trained": len(history["train_loss"]),
    },

    "outputs": {
        "raster_format":  "Cloud Optimized GeoTIFF (COG)",
        "vector_format":  "GeoPackage (GPKG)",
        "vector_layers":  ["buildings (with roof_type: RCC/Tiled/Tin/Others)",
                           "roads",
                           "water_bodies"],
    },

    "class_remap": {
        "original_classes": 9,
        "final_classes":    NUM_CLASSES,
        "rationale": "Railway (4 objects), Bridge (8 objects), Utility points (1227 objects) "
                     "remapped to background due to insufficient pixel coverage (<0.1%) for "
                     "reliable learning. Road and Road_Centre_Line merged into single Road class."
    },
}

report_path = os.path.join(OUT_DIR, "report.json")
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print("📄 SUMMARY REPORT")
print("═" * 50)
print(f"  Model         : {report['model']['architecture']}")
print(f"  Tile size     : {CFG['tile_size']}px")
print(f"  Training tiles: {len(all_imgs)}")
print(f"  Val accuracy  : {best_acc*100:.2f}%")
print(f"  Val mIoU      : {best_miou:.4f}")
print(f"  Raster output : COG GeoTIFF")
print(f"  Vector output : GeoPackage (buildings + roads + water_bodies)")
print(f"  Roof types    : RCC / Tiled / Tin / Others (spectral heuristic)")
print(f"  Report saved  : {report_path}")